In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, initcap, when,
    to_date, coalesce, lit
)
from pyspark.sql.types import StringType

spark = SparkSession.builder \
    .appName("Sales Data Lake Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [3]:
import os
import shutil

# Create Data Lake directories
os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake structure created successfully.")

Data Lake structure created successfully.


In [4]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales.csv


In [5]:
uploaded_filename = list(uploaded.keys())[0]

shutil.copy(
    uploaded_filename,
    "/content/data_lake/bronze/sales.csv"
)

print("Dataset copied to Bronze layer.")

Dataset copied to Bronze layer.


In [6]:
import os

bronze_path = "/content/data_lake/bronze/sales.csv"
silver_path = "/content/data_lake/silver/preprocessed_sales"

print("Bronze file exists:", os.path.exists(bronze_path))
print("Bronze file size:", os.path.getsize(bronze_path), "bytes")

Bronze file exists: True
Bronze file size: 109537 bytes


In [7]:
df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [9]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [10]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [11]:
before_count = df.count()

print("Total records before preprocessing:", before_count)

Total records before preprocessing: 1000


In [12]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Number of exact duplicate rows:", duplicate_count)

Number of exact duplicate rows: 10


In [13]:
df.groupBy(df.columns) \
  .count() \
  .filter(col("count") > 1) \
  .show(truncate=False)

+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Grocery        |7      

In [14]:
from pyspark.sql.functions import sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [15]:
print("Category values:")
df.select("category").distinct().show(truncate=False)

print("City values:")
df.select("city").distinct().show(truncate=False)

Category values:
+---------------+
|category       |
+---------------+
|SPORTS         |
|beauty         |
|Sports         |
|Grocery        |
|Electronics    |
|Clothing       |
|Books          |
|ELECTRONICS    |
|Home Appliances|
|Furniture      |
|sports         |
|Beauty         |
|CLOTHING       |
+---------------+

City values:
+-----------+
|city       |
+-----------+
|delhi      |
| Hyderabad |
|Chennai    |
|Lucknow    |
|Mumbai     |
|DELHI      |
|HYDERABAD  |
|Kolkata    |
|Pune       |
|Delhi      |
|JAIPUR     |
| Mumbai    |
|Coimbatore |
|Bengaluru  |
|Hyderabad  |
|coimbatore |
|Vijayawada |
|Jaipur     |
|NULL       |
+-----------+



In [16]:
invalid_quantity = df.filter(col("quantity") <= 0).count()

invalid_unit_price = df.filter(col("unit_price") <= 0).count()

invalid_discount = df.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).count()

print("Invalid quantity records:", invalid_quantity)
print("Invalid unit price records:", invalid_unit_price)
print("Invalid discount records:", invalid_discount)

Invalid quantity records: 12
Invalid unit price records: 8
Invalid discount records: 8


In [17]:
df.filter(
    (col("quantity") <= 0) |
    (col("unit_price") <= 0) |
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show(truncate=False)

+--------+-----------+----------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100034  |C0192      |Divya Sharma    |Yoga Mat              |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu    |2025-08-19|Delivered   |
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery        |7       |0         |40              |Credit Card     |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100114  |C0252      |Keerthi Singh   |Ceiling Fan           |Home Appliances|0      

In [18]:
df.select("order_date").distinct().show(20, truncate=False)

+----------+
|order_date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2025-12-07|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
+----------+
only showing top 20 rows


In [19]:
silver_df = df.dropDuplicates()

print("Rows after removing duplicates:", silver_df.count())

Rows after removing duplicates: 990


In [20]:
text_columns = [
    "customer_name",
    "payment_method",
    "city"
]

silver_df = silver_df.fillna(
    "Unknown",
    subset=text_columns
)

In [21]:
text_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    silver_df = silver_df.withColumn(
        c,
        trim(col(c))
    )

In [22]:
capitalize_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in capitalize_columns:
    silver_df = silver_df.withColumn(
        c,
        initcap(col(c))
    )

In [23]:
quantity_median = silver_df.filter(
    col("quantity") > 0
).approxQuantile(
    "quantity",
    [0.5],
    0.01
)[0]

print("Median valid quantity:", quantity_median)

Median valid quantity: 5.0


In [24]:
silver_df = silver_df.withColumn(
    "quantity",
    when(
        col("quantity") <= 0,
        lit(quantity_median)
    ).otherwise(col("quantity"))
)

In [25]:
unit_price_median = silver_df.filter(
    col("unit_price") > 0
).approxQuantile(
    "unit_price",
    [0.5],
    0.01
)[0]

print("Median valid unit price:", unit_price_median)

Median valid unit price: 1810.0


In [26]:
silver_df = silver_df.withColumn(
    "unit_price",
    when(
        col("unit_price") <= 0,
        lit(unit_price_median)
    ).otherwise(col("unit_price"))
)

In [27]:
discount_median = silver_df.filter(
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
).approxQuantile(
    "discount_percent",
    [0.5],
    0.01
)[0]

print("Median valid discount:", discount_median)

Median valid discount: 20.0


In [28]:
silver_df = silver_df.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") < 0) |
        (col("discount_percent") > 100),
        lit(discount_median)
    ).otherwise(col("discount_percent"))
)

In [42]:
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

silver_df = df.dropDuplicates()

In [43]:
from pyspark.sql.functions import col, trim, to_date, coalesce

silver_df = silver_df.withColumn(
    "order_date",
    coalesce(
        to_date(trim(col("order_date")), "yyyy-MM-dd"),
        to_date(trim(col("order_date")), "yyyy/MM/dd"),
        to_date(trim(col("order_date")), "MM-dd-yyyy")
    )
)

In [44]:
invalid_date_count = silver_df.filter(
    col("order_date").isNull()
).count()

print("Invalid/unparseable dates after standardization:", invalid_date_count)

Invalid/unparseable dates after standardization: 4


In [45]:
silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [46]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



In [47]:
after_count = silver_df.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)
print("Rows removed:", before_count - after_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [48]:
silver_output_path = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_output_path)

print("Silver data saved successfully.")

Silver data saved successfully.


In [49]:
import glob
import os

csv_files = glob.glob(
    "/content/data_lake/silver/preprocessed_sales/part-*.csv"
)

print(csv_files)

['/content/data_lake/silver/preprocessed_sales/part-00000-0e90d9a4-e62f-49d4-93cb-7a531b3efc2c-c000.csv']


In [50]:
silver_csv = csv_files[0]

final_silver_csv = "/content/data_lake/silver/sales_silver.csv"

shutil.copy(
    silver_csv,
    final_silver_csv
)

print("Silver CSV created:")
print(final_silver_csv)

Silver CSV created:
/content/data_lake/silver/sales_silver.csv


In [51]:
silver_read = spark.read.csv(
    final_silver_csv,
    header=True,
    inferSchema=True
)

silver_read.printSchema()
silver_read.show(10, truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+

In [52]:
silver_read_count = silver_read.count()

print("Silver record count:", silver_read_count)

Silver record count: 990


## Preprocessing Summary

I have performed the following basic preprocessing operations:

1. Loaded the raw sales.csv file from the Bronze layer using Apache Spark.
2. Identified exact duplicate records and removed them.
3. Identified missing values in text columns and replaced them with "Unknown" where appropriate.
4. Trimmed leading and trailing spaces from text fields.
5. Standardized obvious capitalization inconsistencies in text columns.
6. Validated quantity to ensure it was greater than zero. Invalid values were replaced using the median of valid values.
7. Validated unit price to ensure it was greater than zero. Invalid values were replaced using the median of valid values.
8. Validated discount percentage to ensure it was between 0 and 100. Invalid values were replaced using the median of valid values.
9. Converted valid dates from supported formats into a consistent date format.
10. Clearly invalid or unparseable dates were retained as null rather than removing the entire record.
11. Saved the processed dataset in the Silver layer.
12. Read the Silver data back using Spark and verified the resulting record count and sample records.

The Bronze dataset was kept unchanged throughout the preprocessing process.

In [53]:
print("========== FINAL VERIFICATION ==========")
print("Bronze records:", before_count)
print("Silver records:", silver_read_count)
print("Duplicate records removed:", duplicate_count)
print("Invalid quantity records identified:", invalid_quantity)
print("Invalid unit price records identified:", invalid_unit_price)
print("Invalid discount records identified:", invalid_discount)
print("Invalid/unparseable dates identified:", invalid_date_count)
print("========================================")

========== FINAL VERIFICATION ==========
Bronze records: 1000
Silver records: 990
Duplicate records removed: 10
Invalid quantity records identified: 12
Invalid unit price records identified: 8
Invalid discount records identified: 8
Invalid/unparseable dates identified: 4
